# Data validation: a contract your data must pass

_Data Wrangling_

**Write your assumptions down as automated checks, so bad data is caught loud and early.**

You always have assumptions about your data. "Age is between 0 and 120." "Price is never
       negative." "Category is one of books, music, games." "The
       order id is unique." Most of the time these live only in your head.

       A data contract writes those assumptions down as automated checks that run on every
       batch of data. If a new file shows up with an age of 200, a price of -5, or a category nobody has
       seen before, the contract fails loudly right at the door &mdash; instead of letting the bad
       row slip through and silently poison every average, model, and dashboard downstream.

---

This notebook is a practice scaffold for the **Data validation: a contract your data must pass** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q pandera great-expectations
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — pandas + pandera

In [ ]:
import pandas as pd
import pandera as pa
from pandera import Column, Check, DataFrameSchema

# --- A small messy batch (deliberately corrupted to trip the contract) ---
df = pd.DataFrame({
    "order_id":   [1, 2, 3, 4, 12, 12],            # 12 is DUPLICATED
    "age":        [34, 200, 29, -3, 41, 55],       # 200 and -3 are out of range
    "price":      [19.99, 5.00, -2.50, 12.0, 9.9, 9.9],  # -2.50 is negative
    "category":   ["books", "music", "podcasts", "games", "books", "books"],  # 'podcasts' not allowed
    "start_date": pd.to_datetime(["2024-01-01"] * 6),
    "end_date":   pd.to_datetime(["2024-01-05", "2024-01-03", "2023-12-30",   # row 3 ends before it starts
                                  "2024-01-10", "2024-01-07", "2024-01-07"]),
})

ALLOWED = ["books", "music", "games"]

# === The data contract, as a typed schema ===
schema = DataFrameSchema(
    {
        "order_id":   Column(int,   unique=True, nullable=False),            # key: no duplicates
        "age":        Column(float, Check.in_range(0, 120), nullable=False), # range + not-null
        "price":      Column(float, Check.ge(0),            nullable=False), # non-negative + not-null
        "category":   Column(str,   Check.isin(ALLOWED),    nullable=False), # allowed set
        "start_date": Column("datetime64[ns]", nullable=False),
        "end_date":   Column("datetime64[ns]", nullable=False),
    },
    checks=Check(lambda d: d["end_date"] >= d["start_date"],  # cross-field rule
                 error="end_date must be >= start_date"),
    coerce=True,   # try to coerce dtypes; mismatches still surface as errors
)

# lazy=True collects EVERY failure instead of stopping at the first one.
try:
    schema.validate(df, lazy=True)
    print("contract PASSED")
except pa.errors.SchemaErrors as err:
    # err.failure_cases is a tidy DataFrame: column, check, failing value, row index.
    print("contract FAILED:")
    print(err.failure_cases[["column", "check", "failure_case", "index"]])

# === Plain assert checks (no library) — the same intent, minimal ===
assert set(["order_id", "age", "price", "category"]).issubset(df.columns), "missing columns"
assert df["order_id"].is_unique, "order_id has duplicates"
assert df["age"].between(0, 120).all(), "age out of [0, 120]"
assert (df["price"] >= 0).all(), "price is negative"
assert df["category"].isin(ALLOWED).all(), "unexpected category"
assert (df["end_date"] >= df["start_date"]).all(), "end_date before start_date"

# === great_expectations note ===
# For a richer setup, great_expectations expresses the same contract as an
# "Expectation Suite" and auto-generates a validation report + Data Docs:
#   import great_expectations as gx
#   ctx = gx.get_context()
#   batch = ctx.sources.pandas_default.read_dataframe(df)
#   batch.expect_column_values_to_be_between("age", 0, 120)
#   batch.expect_column_values_to_be_in_set("category", ALLOWED)
#   batch.expect_column_values_to_not_be_null("price")
#   batch.expect_column_values_to_be_unique("order_id")
# Run the suite on every new batch in your pipeline; a failed expectation alerts.

## Visualize the data & results

_Run a five-rule data contract on a small messy orders table. How many rows fail each rule? The bars are the contract catching problems._

In [ ]:
import pandas as pd
import numpy as np

# A small messy real-ish orders batch (inline). Corruptions are deliberate.
df = pd.DataFrame({
    "order_id":   [1,2,3,4,5,6,7,8,9,10,11,12,12,14,15,16,17,18,19,20],  # 12 duplicated
    "age":        [34,29,np.nan,52,140,41,23,-3,67,38,45,np.nan,55,200,33,28,49,61,30,22],
    "price":      [19.99,5.00,-2.50,12.0,8.75,np.nan,3.20,15.0,-1.0,7.5,
                   22.0,9.9,9.9,40.0,-5.0,11.1,6.6,np.nan,13.3,2.2],
    "category":   ["books","music","books","TOYS","music","books","games","music",
                   "books","games","music","books","books","unknown","games","music",
                   "books","games","music","books"],
    "start_date": pd.to_datetime(["2024-01-01"]*20),
    "end_date":   pd.to_datetime(["2024-01-05","2024-01-03","2023-12-30","2024-01-10","2024-01-02",
                                  "2024-01-06","2024-01-04","2024-01-01","2023-12-25","2024-01-08",
                                  "2024-01-09","2024-01-07","2024-01-07","2024-01-11","2024-01-03",
                                  "2024-01-05","2024-01-06","2024-01-02","2024-01-04","2024-01-09"]),
})

ALLOWED = {"books", "music", "games"}

# Count rows failing each rule.
n_range = ((df["age"] < 0) | (df["age"] > 120) | (df["price"] < 0)).fillna(False).sum()
n_cat   = (~df["category"].str.lower().isin(ALLOWED)).sum()
n_null  = (df["age"].isna() | df["price"].isna()).sum()
n_dup   = df["order_id"].duplicated(keep=False).sum()
n_cross = (df["end_date"] < df["start_date"]).sum()

print("range     :", int(n_range))  # -> 6
print("category  :", int(n_cat))    # -> 2
print("null      :", int(n_null))   # -> 4
print("uniqueness:", int(n_dup))    # -> 2
print("cross     :", int(n_cross))  # -> 2

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A teammate says "the data looks fine, I eyeballed the first 20 rows." Why is that not a data contract, and what would make it one?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that eyeballing is manual, one-time, and only sees a sample. — _It can't run on every new batch, and it misses problems past the rows you looked at._
- Write the assumptions down as code: schema, ranges, allowed sets, not-null, uniqueness, cross-field rules. — _Executable checks run the same way every time and cover all rows, not just a glanced-at sample._
- Wire those checks into the pipeline so they run automatically on every batch and raise on failure. — _Automation + a loud failure is what turns a good intention into a contract that actually protects you._

**Answer:** Eyeballing is a manual, one-time spot-check &mdash; it sees only a sample and never runs again. A contract is the same intent made into automated checks (schema, ranges, allowed sets, not-null, uniqueness, cross-field) that run on every batch and fail loudly. Tools like pandera or great_expectations, or even plain assert statements, turn the eyeballing into something repeatable and enforceable.

</details>

**Problem 2.** Your age check allows $[0, 999]$ and has never fired. A colleague's allows $[0, 120]$ and fires once a month on a single bad row. Which is the better contract, and what's the risk on each side?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Ask what each bound actually catches. — _$[0, 999]$ admits a 200-year-old customer as "valid", so it catches essentially nothing real._
- Consider the cost of a check that's too tight. — _If $[0, 120]$ fired on legitimate values, it would be a false alarm and people would learn to ignore it._
- Pick the tightest bound that still reflects real, valid data. — _$[0, 120]$ matches actual human ages, so its monthly fire is a real catch, not noise._

**Answer:** The $[0, 120]$ check is far better: it reflects real ages and its monthly fire is a genuine catch. The $[0, 999]$ check is too loose &mdash; it never fires because it never can, so it protects nothing. The risk on the other side is a bound so tight it flags valid data and trains people to ignore the alert. Aim for the tightest bound that still admits all legitimate values.

</details>

**Problem 3.** An upstream team renames category to cat and adds a new value podcasts. Which parts of the contract should fire, and is firing a bug or a feature here?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check the schema rule. — _The expected column category is now missing, so the schema/column check fails immediately._
- Check the allowed-set rule. — _Even if you map the new name, podcasts is not in $\{$books, music, games$\}$, so the category check fails on those rows._
- Decide whether the change was intended. — _If it was an upstream mistake, the contract just saved you; if it was a real, agreed change, you update the contract deliberately._

**Answer:** The schema check fires (the category column is gone) and, once mapped, the allowed-set check fires on the podcasts rows. This is the contract working, not a bug &mdash; it caught schema drift. If the change was a mistake, you've avoided silent corruption; if it was a legitimate, agreed change, you update the contract on purpose to add the new column name and value.

</details>